In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from pprint import pprint

from glide.common_components.utils import mask_average
from glide.common_components.utils import circular_mask
from glide.common_components import constants
import glide.science_data_processing.L1A as L1A
import glide.common_components.view_geometry as view_geometry
from glide.common_components.stars import get_beta_angle

In [ ]:
imager = 'WFI'
top_col_biases = np.load(f'products/COL_BIAS_{imager}_TOP.npy')
bottom_col_biases = np.load(f'products/COL_BIAS_{imager}_BOTTOM.npy')
half_npix = constants.NPIX[imager] // 2
top_correction =  top_col_biases[np.newaxis, np.newaxis, :]
bottom_correction =  bottom_col_biases[np.newaxis, np.newaxis, :]

In [ ]:
oob_filepath = "/data/L1A/CARRUTHERS_GCI-WFI_L1A-STR_20251221_v1.0.nc"

with xr.open_dataset(oob_filepath) as ds_oob:
    l1a_str = L1A.L1A(ds_oob)
#print(dir(l1a_str))

str_ims = l1a_str.images
n_frames = l1a_str.n_frames
str_ims = str_ims / n_frames[:, np.newaxis, np.newaxis]

# Subtract voltage biases
str_ims[:, :half_npix, :] -= top_correction
str_ims[:, half_npix:, :] -= bottom_correction

for i, image in enumerate(str_ims):
    
    plt.imshow(image, vmax=np.percentile(image, 99), vmin=0)
    plt.title(f"{imager}_L1A-STR {l1a_str.filters[i]}, DN/frame\n{l1a_str.time[i]}")
    plt.colorbar()
    plt.show()
